# Aprendizaje Supervisado

Aplicamos tra transformaciones que utilizamos en EDA.ipynb

## Holdout: S1-alt

**S1-alt queda reservado para el test final.** Todo el tuneo, la elección de reducción, `GroupedPCA`, LOSO y umbral se hacen sin S1-alt. S0 sigue en entrenamiento; la especificidad S0 que se reporta es la **out-of-fold** del LOSO de tuning, no la del modelo final.


In [2]:
# --- Configuración: una sola ruta, sin absolutos ---
import sys
from pathlib import Path
import pandas as pd
from src2.data import eliminar_columnas, imputar_media
from src2.features import rango_movil
import src2.train as train
import src2.split as split


RUTA = Path.cwd()
if not (RUTA / "datasets").exists():
    RUTA = RUTA.parent
sys.path.insert(0, str(RUTA / "Colab"))
BASE = RUTA / "datasets"


print("raíz:", RUTA)
print("leyendo CSVs...")
dfs = {
    "S0":     pd.read_csv(BASE / "datasetS0.csv"),
    "S1":     pd.read_csv(BASE / "datasetS1.csv"),
    "S1-alt": pd.read_csv(BASE / "datasetS1alt.csv"),
    "S2":     pd.read_csv(BASE / "datasetS2.csv"),
    "S3":     pd.read_csv(BASE / "datasetS3.csv"),
    "S4":     pd.read_csv(BASE / "datasetS4.csv"),
    "S5":     pd.read_csv(BASE / "datasetS5.csv"),
    "S6":     pd.read_csv(BASE / "datasetS6.csv"),
}

# Agregar el escenario a cada bloque antes de concatenar evita el
# DataFrame fragmentado que generaba el MultiIndex + reset_index.
df = (pd.concat(dfs, names=["escenario"])
      .reset_index(level="escenario")
      .reset_index(drop=True))

df = imputar_media(eliminar_columnas(df))


raíz: /media/windows/Users/pablo/Documents/UNSA/4to/MetodosCuantitativos/TPFinal/tpfinal-colab
leyendo CSVs...


/tmp/ipykernel_3751963/1607311775.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index(level="escenario")


## 1. Comparación de transformaciones

**Objetivo.** Elegir la representación de entrada dejando fijo el modelo. Las tres variantes usan el mismo Random Forest (`PARAMS_RF`) y el mismo LOSO (`LeaveOneGroupOut`); así la diferencia se atribuye a la transformación y no al clasificador.

- **Niveles (crudo).** Cada fila es un segundo con las mediciones tal como llegan. Conserva el nivel de operación; el riesgo es que el modelo aprenda la hora o el escenario en lugar del ataque.
- **Rango móvil (120 s).** Máximo menos mínimo en una ventana de dos minutos, dentro de cada escenario. Quita el nivel y conserva la amplitud local; es invariante a sumar una constante. Es la representación que mejor separó en el EDA.
- **Ventana de 20 pasos.** Una muestra son 20 segundos aplanados: 132 × 20 = 2640 features. La idea se toma de GRIDGUARD-AI, que evalúa ventanas de 20 pasos con 141 variables; acá se replica sobre nuestras columnas. Agrega contexto temporal, pero pierde las primeras 19 filas de cada escenario y hereda la etiqueta de la última fila de la ventana.

**Métricas.** Accuracy, F1, recall@5 y especificidad en S0. No usamos AUC como criterio principal.


In [3]:
ESCENARIO_TEST = "S1-alt"

df_test = df[df["escenario"] == ESCENARIO_TEST].reset_index(drop=True)
df_train = df[df["escenario"] != ESCENARIO_TEST].reset_index(drop=True)

dfX = df_train.drop(["attack_active"], axis=1)
dfY = df_train["attack_active"]
dfX_test = df_test.drop(["attack_active"], axis=1)
dfY_test = df_test["attack_active"]

print("train:", sorted(df_train["escenario"].unique()))
print("test:", sorted(df_test["escenario"].unique()))

train.rf_logo(dfX.drop(columns="escenario"), dfY, dfX["escenario"])


train: ['S0', 'S1', 'S2', 'S3', 'S4', 'S5', 'S6']
test: ['S1-alt']


,escenario,features,accuracy_train,n,positivos,accuracy,precision,recall,f1,especificidad,auc,recall_5
fold,,,,,,,,,,,,
1,S0,132,0.998,1798.000,0.000,0.349,0.000,0.000,0.000,0.349,NaN,NaN
2,S1,132,0.998,1798.000,913.000,0.492,0.000,0.000,0.000,1.000,0.745,0.337
3,S2,132,0.998,1768.000,903.000,0.610,0.672,0.461,0.547,0.765,0.754,0.000
4,S3,132,0.998,1800.000,912.000,0.507,0.719,0.045,0.085,0.982,0.828,0.333
5,S4,132,0.999,1798.000,1257.000,0.645,0.940,0.526,0.674,0.922,0.898,0.440
6,S5,132,0.998,1800.000,1341.000,0.511,0.907,0.383,0.539,0.885,0.824,0.022
7,S6,132,0.998,1800.000,1350.000,0.363,0.943,0.161,0.275,0.971,0.898,0.273
mean,NaN,132,0.998,1794.571,953.714,0.497,0.597,0.225,0.303,0.839,0.824,0.234


In [4]:
def rango_por_escenario(X):
    """Rango móvil de 120 s, escenario por escenario."""
    partes = []
    for k in X["escenario"].unique():
        bloque = X.loc[X["escenario"] == k].drop(columns="escenario")
        partes.append(rango_movil(bloque, 120))
    return pd.concat(partes, ignore_index=True)

X_rango = rango_por_escenario(dfX)
X_rango_test = rango_por_escenario(dfX_test)

print("rango train:", X_rango.shape)
print("rango test:", X_rango_test.shape)


rango train: (12562, 132)
rango test: (1792, 132)


In [5]:
X_win, Y_win = split.aplanar_df(dfX, dfY,20)
#train.rf_logo(X_win.drop("escenario", axis=1), Y_win, X_win["escenario"])

In [7]:
# === 6. La comparación que faltaba: block PCA sobre rango ===
from src2.features import rango_movil

# --- Las cuatro configuraciones: niveles/rango × sin/block ---
tablas = {
    "Crudo": train.rf_logo(dfX.drop(columns="escenario"),dfY,dfX["escenario"]),
    "rango movil": train.rf_logo(X_rango, dfY, dfX["escenario"]),
    "ventana deslizante": train.rf_logo(X_win.drop("escenario", axis=1), Y_win, X_win["escenario"]),
}
# --- Resumen: ranking (AUC, recall@5) y punto de operación (accuracy, balanced acc) ---
filas = []
for nombre, tabla in tablas.items():
    d = tabla.drop(index="mean", errors="ignore")
    filas.append({
        "configuración": nombre,
        "features": int(d["features"].iloc[0]),
        "AUC media": d["auc"].mean(),
        "recall@5": d["recall_5"].mean(),
        "recall": d["recall"].mean(),
        "espec S0": float(d.loc[d.escenario == "S0", "especificidad"].iloc[0]),
        "accuracy train": d["accuracy_train"].mean(),
        "accuracy test": d["accuracy"].mean(),
        "balanced acc": ((d["recall"] + d["especificidad"]) / 2).mean(),
    })
resumen = pd.DataFrame(filas).set_index("configuración")
display(resumen.round(3))

KeyboardInterrupt: 

**Nota sobre las métricas.** La accuracy y el F1 se calculan con umbral 0.5 sobre la probabilidad del Random Forest. `recall@5` se calcula con un 5 % de falsos positivos (equivale a 95 % de especificidad). La especificidad se informa en S0, el escenario sin ataques; en ese fold no hay positivos, así que recall y AUC no se calculan. No usamos AUC como criterio de decisión. Estas métricas describen un punto de operación concreto; no son propiedades libres de umbral del modelo.


### Lectura

- **Niveles** conserva el nivel de operación: el RF puede aprender la hora o el escenario. Es la representación más frágil.
- **Rango** quita el nivel y conserva la dinámica local. Es la que mejor rinde en F1/accuracy y la que se usa en el resto del trabajo.
- **Ventana 20** agrega contexto temporal (GRIDGUARD-AI), pero cuesta 2640 features y pierde 19 filas por escenario. Queda como contraste, no como representación principal.
- La especificidad en S0 sigue siendo baja en las tres: la representación no arregla la calibración.


## 2. Comparación de reducciones

**Objetivo.** Elegir la reducción con la representación fija (**rango**) y el mismo Random Forest (`PARAMS_RF`). Cada reductor se ajusta dentro de cada fold LOSO, así que no ve el escenario de test.

Métodos:

- **ninguna:** las 132 columnas de rango.
- **`corr`:** descarta una columna de cada par con |r| > 0.95.
- **`pca`:** estandariza y conserva el 95 % de la varianza.
- **`block_pca`:** agrupa por correlación y aplica PCA por bloque (`GroupedPCA`).
- **RFE:** elimina columnas de a 5 y prueba 10, 20, 40 y 80. Va al final porque es el más caro.

Mismas métricas y misma nota que en la sección 1: accuracy y F1 a umbral 0.5, recall@5 a 5 % de falsos positivos, especificidad en S0.


In [13]:
# === 2. Reducciones rápidas sobre rango, con Random Forest fijo ===
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from src2.group import GroupedPCA
from src2.reduce import CorrFilter
from src2.search import rf_pipeline_logo
from src2.train import PARAMS_RF, SEED

rf = lambda: RandomForestClassifier(**PARAMS_RF)

pipelines = {
    "ninguna": Pipeline([("modelo", rf())]),
    "corr": Pipeline([("corr", CorrFilter()), ("modelo", rf())]),
    "pca": Pipeline([
        ("scale", StandardScaler()),
        ("pca", PCA(n_components=0.95, random_state=SEED)),
        ("modelo", rf()),
    ]),
    "block_pca": Pipeline([("block", GroupedPCA()), ("modelo", rf())]),
}

tablas_reduccion = {
    nombre: rf_pipeline_logo(pipe, X_rango, dfY, dfX["escenario"])
    for nombre, pipe in pipelines.items()
}

def resumen_reduccion(nombre, tabla):
    d = tabla.drop(index="mean")
    ataque = d[d["escenario"] != "S0"]
    return {
        "reducción": nombre,
        "features": int(d["features"].iloc[0]),
         "accuracy train": d["accuracy_train"].mean(),
        "accuracy test": d["accuracy"].mean(),
        "F1": ataque["f1"].mean(),
        "recall@5": ataque["recall_5"].mean(),
        "recall": ataque["recall"].mean(),
        "espec S0": float(d.loc[d["escenario"] == "S0", "especificidad"].iloc[0]),
    }

comparacion_reduccion = pd.DataFrame([
    resumen_reduccion(nombre, tabla)
    for nombre, tabla in tablas_reduccion.items()
]).set_index("reducción")

display(comparacion_reduccion.round(3))


,features,accuracy train,accuracy test,F1,recall@5,recall,espec S0
reducción,,,,,,,
ninguna,132,0.999,0.719,0.722,0.457,0.626,0.561
corr,50,0.999,0.701,0.693,0.491,0.604,0.507
pca,19,0.999,0.690,0.712,0.398,0.641,0.417
block_pca,37,0.999,0.705,0.693,0.438,0.629,0.461


### 2.1 RFE 

RFE se ajusta dentro de cada fold: en cada vuelta elimina las columnas menos importantes y reentrena. La grilla prueba 10, 20, 40 y 80 columnas con `step=5`.

Primero se elige la cantidad de columnas con accuracy LOSO; después se evalúa ese pipeline fold por fold con las mismas métricas. Si tarda demasiado, subí `step` a 10 o reducí la lista de `n_features`.


In [11]:
from sklearn.ensemble import RandomForestClassifier
import src2.search as search
from src2.search import rf_pipeline_logo
from src2.train import PARAMS_RF


def resumen_reduccion(nombre, tabla):
    d = tabla.drop(index="mean", errors="ignore")
    ataque = d[d["escenario"] != "S0"]
    return {
        "reducción": nombre,
        "features": int(d["features"].iloc[0]),
        "accuracy train": d["accuracy_train"].mean(),
        "accuracy test": d["accuracy"].mean(),
        "F1": ataque["f1"].mean(),
        "recall@5": ataque["recall_5"].mean(),
        "recall": ataque["recall"].mean(),
        "espec S0": float(d.loc[d["escenario"] == "S0", "especificidad"].iloc[0]),
    }


# 1. Reconstruir la comparación si este kernel no la tiene
if "comparacion_reduccion" not in globals():
    if "tablas_reduccion" in globals():
        comparacion_reduccion = pd.DataFrame([
            resumen_reduccion(nombre, tabla)
            for nombre, tabla in tablas_reduccion.items()
        ]).set_index("reducción")
    else:
        comparacion_reduccion = pd.DataFrame(
            columns=["features", "accuracy train", "accuracy test",
                     "F1", "recall@5", "recall", "espec S0"]
        )

# 2. RFE
X_rfe = X_rango

busqueda_rfe = search.buscar_grid_rfe(
    X_rfe,
    dfY,
    dfX["escenario"],
    RandomForestClassifier(**PARAMS_RF),
    n_features=[10, 20, 40, 80],
    step=5,
    scoring="accuracy",
)

print("mejor cantidad de columnas:", busqueda_rfe.best_params_["rfe__n_features_to_select"])
display(search.tabla_resultados(busqueda_rfe).round(4))

# 3. Columnas seleccionadas
rfe = busqueda_rfe.best_estimator_.named_steps["rfe"]
columnas_rfe = X_rfe.columns[rfe.support_]

print("columnas seleccionadas:", len(columnas_rfe))
display(pd.DataFrame({"columna": columnas_rfe}))

# 4. Evaluar el pipeline elegido con LOSO
tabla_rfe = rf_pipeline_logo(
    busqueda_rfe.best_estimator_,
    X_rfe,
    dfY,
    dfX["escenario"],
)
fila_rfe = resumen_reduccion("RFE", tabla_rfe)

# 5. Agregar la fila sin duplicarla
comparacion_reduccion = comparacion_reduccion.drop(index="RFE", errors="ignore")
comparacion_reduccion = pd.concat([
    comparacion_reduccion,
    pd.DataFrame([fila_rfe]).set_index("reducción"),
])
display(comparacion_reduccion.round(3))

Fitting 8 folds for each of 4 candidates, totalling 32 fits


/home/danilo/Windows_Compartido/Mis documentos/UNSA/4to/MetodosCuantitativos/TPFinal/tpfinal-colab/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/danilo/Windows_Compartido/Mis documentos/UNSA/4to/MetodosCuantitativos/TPFinal/tpfinal-colab/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/danilo/Windows_Compartido/Mis documentos/UNSA/4to/MetodosCuantitativos/TPFinal/tpfinal-colab/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should 

mejor cantidad de columnas: 80


,param_rfe__n_features_to_select,mean_test_score,std_test_score,rank_test_score
3,80,0.6993,0.2226,1
2,40,0.6971,0.2180,2
1,20,0.6967,0.2134,3
0,10,0.6923,0.1945,4


columnas seleccionadas: 80


,columna
0,BSS-Inverter-AC-Meter.cos_phi1
1,BSS-Inverter-AC-Meter.cos_phi2
2,BSS-Inverter-AC-Meter.cos_phi3
3,BSS-Inverter-AC-Meter.freq
4,BSS-Inverter-AC-Meter.i_sum3
...,...
75,Substation-AC-Meter.qln1
76,Substation-AC-Meter.qln2
77,Substation-AC-Meter.s_sum3
78,Substation-AC-Meter.sln3


,features,accuracy train,accuracy test,F1,recall@5,recall,espec S0
RFE,80,0.98675,0.699375,0.698714,0.407571,0.689857,0.397


### Lectura

- **`corr` y `block_pca`** son las alternativas razonables: descartan redundancia sin mezclar variables distintas. Según la evidencia previa, `corr` gana en AUC/recall@5 y `block_pca` es más compacta.
- **`pca` global** suele perder porque un árbol corta una variable a la vez y las componentes son mezclas de las 132 columnas.
- **RFE** no garantiza mejorar y es el más caro; su selección puede cambiar entre folds.
- La decisión no debería depender de una décima de AUC: mirá accuracy/F1, recall@5 y especificidad S0 en conjunto.


Justificacion de eleccion de block PCA
Elegimos blockPCA por que da mejores resultaado y realizaremos el tuneo de parametros a partir de este

In [7]:
from sklearn.ensemble import RandomForestClassifier
from src2.search import tunear_clasificador
from src2.train import PARAMS_RF

espacio_rf = {
    "modelo__n_estimators": [100, 200, 400],
    "modelo__max_depth": [5, 10, 18, None],
    "modelo__min_samples_leaf": [1, 2, 4, 8],
    "modelo__max_features": ["sqrt", "log2", 0.3, 0.5],
}
resultado_rf = tunear_clasificador(
    X_rango,
    dfY,
    dfX["escenario"],
    RandomForestClassifier(**{**PARAMS_RF, "n_jobs": 1}),
    espacio_rf,
    "RF + block_pca (tuneado)",
    n_iter=30,
    tolerancia=0.02,
)

display(resultado_rf["fila"].round(3))
display(resultado_rf["elegido"]["params"])

Fitting 7 folds for each of 30 candidates, totalling 210 fits


,features,accuracy train,accuracy test,gap,F1,recall@5,recall,espec S0
configuración,,,,,,,,
RF + block_pca (tuneado),37,0.949,0.733,0.215,0.708,0.498,0.665,0.519


{'modelo__n_estimators': np.int64(200),
 'modelo__min_samples_leaf': np.int64(2),
 'modelo__max_features': 'log2',
 'modelo__max_depth': 5}

In [14]:
from xgboost import XGBClassifier
from src2.train import PARAMS_XGB

espacio_xgb = {
    "modelo__n_estimators": [100, 300, 500],
    "modelo__max_depth": [3, 5, 7],
    "modelo__learning_rate": [0.01, 0.05, 0.1],
    "modelo__subsample": [0.7, 0.9, 1.0],
}

resultado_xgb = tunear_clasificador(
    X_rango, dfY, dfX["escenario"],
    XGBClassifier(**{**PARAMS_XGB, "n_jobs": 1}),
    espacio_xgb,
    "XGB + block_pca (tuneado)",
    n_iter=20,
)

Fitting 7 folds for each of 20 candidates, totalling 140 fits


In [12]:
display(resultado_xgb["fila"].round(3))
display(resultado_xgb["elegido"]["params"])

,features,accuracy train,accuracy test,gap,F1,recall@5,recall,espec S0
configuración,,,,,,,,
XGB + block_pca (tuneado),37,0.914,0.72,0.194,0.689,0.479,0.637,0.537


{'modelo__subsample': np.float64(0.7),
 'modelo__n_estimators': np.int64(100),
 'modelo__max_depth': np.int64(3),
 'modelo__learning_rate': np.float64(0.01)}

In [15]:
from catboost import CatBoostClassifier
from src2.search import tunear_clasificador
from src2.train import SEED

PARAMS_CAT = {
    "iterations": 500,
    "depth": 6,
    "learning_rate": 0.05,
    "l2_leaf_reg": 3,
    "loss_function": "Logloss",
    "eval_metric": "Logloss",
    "auto_class_weights": "Balanced",
    "random_seed": SEED,
    "thread_count": 1,
    "verbose": False,
    "allow_writing_files": False,
}

espacio_cat = {
    "modelo__iterations": [300, 500, 800],
    "modelo__depth": [4, 6, 8],
    "modelo__learning_rate": [0.01, 0.05, 0.1],
    "modelo__l2_leaf_reg": [1, 3, 10],
}

resultado_cat = tunear_clasificador(
    X_rango,
    dfY,
    dfX["escenario"],
    CatBoostClassifier(**PARAMS_CAT),
    espacio_cat,
    "CatBoost + block_pca (tuneado)",
    n_iter=20,
)

display(resultado_cat["fila"].round(3))
display(resultado_cat["elegido"]["params"])

Fitting 7 folds for each of 20 candidates, totalling 140 fits


,features,accuracy train,accuracy test,gap,F1,recall@5,recall,espec S0
configuración,,,,,,,,
CatBoost + block_pca (tuneado),37,0.984,0.706,0.278,0.672,0.425,0.61,0.511


{'modelo__learning_rate': np.float64(0.01),
 'modelo__l2_leaf_reg': np.int64(1),
 'modelo__iterations': np.int64(300),
 'modelo__depth': np.int64(4)}

## 3. Test final sobre S1-alt

S1-alt no participó del tuneo. El pipeline elegido se reentrena con **todo el train** y se evalúa una sola vez acá. `espec S0` se toma out-of-fold del LOSO de tuning; S0 sigue en entrenamiento.


In [16]:
# === Test final: S1-alt ===
from sklearn.base import clone
from src2.train import _metricas

def evaluar_holdout(nombre, resultado):
    modelo = clone(resultado["elegido"]["pipeline"]).fit(X_rango, dfY.to_numpy())
    proba = modelo.predict_proba(X_rango_test.reset_index(drop=True))[:, 1]
    m = _metricas(dfY_test.to_numpy(), proba)

    tabla = resultado["tabla"]
    espec_s0 = float(tabla.loc[tabla["escenario"] == "S0", "especificidad"].iloc[0])

    return {
        "configuración": nombre,
        "features": int(modelo.named_steps["modelo"].n_features_in_),
        "AUC media": m["auc"],
        "recall@5": m["recall_5"],
        "recall": m["recall"],
        "espec S0": espec_s0,
        "accuracy train": modelo.score(X_rango, dfY.to_numpy()),
        "accuracy test": m["accuracy"],
        "balanced acc": (m["recall"] + m["especificidad"]) / 2,
    }

filas_holdout = []
for nombre, variable in (
    ("RF + block_pca", "resultado_rf"),
    ("XGB + block_pca", "resultado_xgb"),
    ("CatBoost + block_pca", "resultado_cat"),
):
    resultado = globals().get(variable)
    if resultado is not None:
        filas_holdout.append(evaluar_holdout(nombre + " (S1-alt)", resultado))

if filas_holdout:
    display(pd.DataFrame(filas_holdout).set_index("configuración").round(3))
else:
    print("No hay resultados de tuneo en memoria. Corré primero RF, XGB o CatBoost.")


,features,AUC media,recall@5,recall,espec S0,accuracy train,accuracy test,balanced acc
configuración,,,,,,,,
RF + block_pca (S1-alt),37,0.881,0.596,0.926,0.519,0.930,0.777,0.775
XGB + block_pca (S1-alt),37,0.783,0.142,0.801,0.537,0.909,0.724,0.723
CatBoost + block_pca (S1-alt),37,0.884,0.592,0.823,0.511,0.975,0.749,0.748


In [ ]:
# GridSearch

In [8]:
from sklearn.ensemble import RandomForestClassifier
from src2.search import tunear_clasificador
from src2.train import PARAMS_RF

espacio_rf_grid = {
    "modelo__n_estimators": [150, 200, 250, 300],
    "modelo__max_depth": [3, 4, 5, 6,7],
    "modelo__min_samples_leaf": [1,2, 4],
    "modelo__max_features": ["log2", "sqrt", 0.15, 0.25],
}

resultado_rf_grid = tunear_clasificador(
    X_rango,
    dfY,
    dfX["escenario"],
    RandomForestClassifier(**{**PARAMS_RF, "n_jobs": 1}),
    espacio_rf_grid,
    "RF + block_pca (grid local)",
    modo="grid",
    tolerancia=0.02,
)

display(resultado_rf_grid["fila"].round(3))
display(resultado_rf_grid["elegido"]["params"])

Fitting 7 folds for each of 240 candidates, totalling 1680 fits


,features,accuracy train,accuracy test,gap,F1,recall@5,recall,espec S0
configuración,,,,,,,,
RF + block_pca (grid local),37,0.884,0.729,0.155,0.706,0.464,0.656,0.512


{'modelo__max_depth': np.int64(3),
 'modelo__max_features': 'log2',
 'modelo__min_samples_leaf': np.int64(1),
 'modelo__n_estimators': np.int64(300)}

In [9]:
from xgboost import XGBClassifier
from src2.search import tunear_clasificador
from src2.train import PARAMS_XGB

espacio_xgb_grid = {
    "modelo__n_estimators": [50, 100, 150, 200],
    "modelo__max_depth": [2, 3, 4],
    "modelo__learning_rate": [0.005, 0.01, 0.03],
    "modelo__subsample": [0.5, 0.6, 0.7, 0.8],
    
}

resultado_xgb_grid = tunear_clasificador(
    X_rango,
    dfY,
    dfX["escenario"],
    XGBClassifier(**{**PARAMS_XGB, "n_jobs": 1}),
    espacio_xgb_grid,
    "XGB + block_pca (grid local)",
    modo="grid",
    tolerancia=0.02,
)

display(resultado_xgb_grid["fila"].round(3))
display(resultado_xgb_grid["elegido"]["params"])

Fitting 7 folds for each of 144 candidates, totalling 1008 fits


,features,accuracy train,accuracy test,gap,F1,recall@5,recall,espec S0
configuración,,,,,,,,
XGB + block_pca (grid local),37,0.862,0.71,0.152,0.677,0.51,0.608,0.556


{'modelo__learning_rate': np.float64(0.01),
 'modelo__max_depth': np.int64(2),
 'modelo__n_estimators': np.int64(50),
 'modelo__subsample': np.float64(0.5)}

In [10]:
from catboost import CatBoostClassifier
from src2.search import tunear_clasificador
from src2.train import SEED

PARAMS_CAT = {
    "iterations": 500,
    "depth": 6,
    "learning_rate": 0.05,
    "l2_leaf_reg": 3,
    "loss_function": "Logloss",
    "eval_metric": "Logloss",
    "auto_class_weights": "Balanced",
    "random_seed": SEED,
    "thread_count": 1,
    "verbose": False,
    "allow_writing_files": False,
}

espacio_cat_grid = {
    "modelo__iterations": [100, 200, 300, 400],
    "modelo__depth": [2, 3, 4, 5],
    "modelo__learning_rate": [0.005, 0.01, 0.03],
    "modelo__l2_leaf_reg": [0.1, 0.5, 1, 2],
}

resultado_cat_grid = tunear_clasificador(
    X_rango,
    dfY,
    dfX["escenario"],
    CatBoostClassifier(**PARAMS_CAT),
    espacio_cat_grid,
    "CatBoost + block_pca (grid local)",
    modo="grid",
    tolerancia=0.02,
)

display(resultado_cat_grid["fila"].round(3))
display(resultado_cat_grid["elegido"]["params"])

Fitting 7 folds for each of 192 candidates, totalling 1344 fits


,features,accuracy train,accuracy test,gap,F1,recall@5,recall,espec S0
configuración,,,,,,,,
CatBoost + block_pca (grid local),37,0.848,0.729,0.119,0.693,0.414,0.638,0.56


{'modelo__depth': np.int64(2),
 'modelo__iterations': np.int64(100),
 'modelo__l2_leaf_reg': np.float64(0.5),
 'modelo__learning_rate': np.float64(0.005)}

# Test contra S1-alt

In [13]:
# === Test final: S1-alt ===
from sklearn.base import clone
from src2.train import _metricas

def evaluar_holdout(nombre, resultado):
    modelo = clone(resultado["elegido"]["pipeline"]).fit(X_rango, dfY.to_numpy())
    proba = modelo.predict_proba(X_rango_test.reset_index(drop=True))[:, 1]
    m = _metricas(dfY_test.to_numpy(), proba)

    tabla = resultado["tabla"]
    espec_s0 = float(tabla.loc[tabla["escenario"] == "S0", "especificidad"].iloc[0])

    return {
        "configuración": nombre,
        "features": int(modelo.named_steps["modelo"].n_features_in_),
        "AUC media": m["auc"],
        "recall@5": m["recall_5"],
        "recall": m["recall"],
        "espec S0": espec_s0,
        "accuracy train": modelo.score(X_rango, dfY.to_numpy()),
        "accuracy test": m["accuracy"],
        "balanced acc": (m["recall"] + m["especificidad"]) / 2,
    }

filas_holdout = []
for nombre, variable in (
    ("RF + block_pca (random)", "resultado_rf"),
    ("XGB + block_pca (random)", "resultado_xgb"),
    ("CatBoost + block_pca (random)", "resultado_cat"),
    ("RF + block_pca (grid)", "resultado_rf_grid"),
    ("XGB + block_pca (grid)", "resultado_xgb_grid"),
    ("CatBoost + block_pca (grid)", "resultado_cat_grid"),
):
    resultado = globals().get(variable)
    if resultado is not None:
        filas_holdout.append(evaluar_holdout(nombre + " (S1-alt)", resultado))

if filas_holdout:
    display(pd.DataFrame(filas_holdout).set_index("configuración").round(6))
else:
    print("No hay resultados de tuneo en memoria. Corré primero RF, XGB o CatBoost.")


,features,AUC media,recall@5,recall,espec S0,accuracy train,accuracy test,balanced acc
configuración,,,,,,,,
RF + block_pca (random) (S1-alt),37,0.881140,0.595604,0.926374,0.519,0.930107,0.777344,0.774978
RF + block_pca (grid) (S1-alt),37,0.868138,0.481319,0.931868,0.512,0.878125,0.742188,0.739177
XGB + block_pca (grid) (S1-alt),37,0.833944,0.205605,0.976923,0.556,0.850820,0.804129,0.801387
CatBoost + block_pca (grid) (S1-alt),37,0.742735,0.057143,0.790110,0.560,0.824869,0.725446,0.724420
